### MiddleWare

In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import HumanMessage, AIMessage
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from pprint import pprint
from typing import Any

Globals

In [2]:
model = f"bedrock:us.anthropic.claude-opus-4-6-v1"

In [3]:
agent = create_agent(
    model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [4]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

for m in response['messages']:
    m.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
The user is engaging in a playful, fictional worldbuilding Q&A about a made-up lunar civilization.

## SUMMARY
We established the following fictional details about the moon:
- Capital: Lunapolis
- Weather in Lunapolis: Clear skies, high of 120°C, low of -100°C
- 100,000 cheese miners live in Lunapolis
- The cheese miners' union is likely to strike due to dissatisfaction with the new president

## ARTIFACTS
None

## NEXT STEPS
Continue answering the user's questions about this fictional lunar civilization, maintaining consistency with established details.
================================ Human Message =================================

If you were Lunapolis' new president how would you respond to the cheese miners' union?
================================== Ai Message ==================================

What a fun challenge! Okay, let me step

Trim Messages Middleware Function

In [5]:
@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [6]:
agent = create_agent(
    model,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [7]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

for m in response['messages']:
    m.pretty_print()

================================ Human Message =================================

My device won't turn on. What should I do?
================================== Ai Message ==================================

Is the device plugged in and turned on?
================================ Human Message =================================

Yes, it's plugged in and turned on.
================================== Ai Message ==================================

Is the device showing any lights or indicators?
================================ Human Message =================================

What's the temperature of the device?
================================== Ai Message ==================================

That's a good question to consider. If the device is overheating, it could cause it to shut down or not turn on properly. Here are a few things to check:

1. **Feel the device** - Is it unusually hot, warm, or cool to the touch?
2. **Check for ventilation** - Make sure the vents aren't blocked by dust,